**This is the analysis part used for the data gathered during the participant experiment**

In [128]:
#imports
import numpy
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import csv
from pathlib import Path
from IPython.display import display
import analysisfunctions as af

In [129]:
#import data from CSV files
df_eye = pd.read_csv("aggregated_reading_eye_metrics.csv")
df_q = pd.read_csv("aggregated_questionnaire_metrics.csv")

*Check that data imports worked*

In [130]:
issues = []

# ── 1. SHAPE ────────────────────────────────────────────────
print("=== SHAPE ===")
print(f"Questionnaire : {df_q.shape[0]} rows × {df_q.shape[1]} cols")
print(f"Eye metrics   : {df_eye.shape[0]} rows × {df_eye.shape[1]} cols")

# Questionnaire: 1 row per participant → expect 12 rows
# Eye metrics:   3 rows per participant (one per condition) → expect N×3
if df_q.shape[0] != df_q["Participant ID"].nunique():
    issues.append("Questionnaire has duplicate participant rows")

if df_eye.shape[0] % 3 != 0:
    issues.append(f"Eye metrics row count ({df_eye.shape[0]}) is not divisible by 3 — missing condition rows?")


# ── 2. PARTICIPANT ID ALIGNMENT ──────────────────────────────
print("\n=== PARTICIPANT ID ALIGNMENT ===")
q_pids   = set(df_q["Participant ID"].unique())
eye_pids = set(df_eye["Participant ID"].unique())

only_in_q   = q_pids - eye_pids
only_in_eye = eye_pids - q_pids

if only_in_q:
    issues.append(f"Participants in questionnaire but NOT in eye metrics: {sorted(only_in_q)}")
if only_in_eye:
    issues.append(f"Participants in eye metrics but NOT in questionnaire: {sorted(only_in_eye)}")

shared_pids = q_pids & eye_pids
print(f"Shared participants : {len(shared_pids)}")
print(f"Only in questionnaire : {sorted(only_in_q) or 'none'}")
print(f"Only in eye metrics   : {sorted(only_in_eye) or 'none'}")


# ── 3. CONDITION COMPLETENESS (eye metrics) ──────────────────
print("\n=== CONDITION COMPLETENESS ===")
EXPECTED_CONDITIONS = {"none", "full", "eyetracked"}

for pid, grp in df_eye.groupby("Participant ID"):
    found = set(grp["Condition"].dropna().unique())
    missing = EXPECTED_CONDITIONS - found
    if missing:
        issues.append(f"{pid} missing conditions in eye metrics: {missing}")

print("Condition counts per participant:")
print(df_eye.groupby("Participant ID")["Condition"].value_counts().unstack(fill_value=0))


# ── 4. CONDITION LABEL CONSISTENCY ──────────────────────────
print("\n=== CONDITION LABELS ===")
# Questionnaire uses: "none" / "full" / "eyetracked" (in column names as "Bees"/"Hubble"/"Salt")
# Eye metrics Condition column should only contain the three expected values
unexpected_conditions = set(df_eye["Condition"].dropna().unique()) - EXPECTED_CONDITIONS
if unexpected_conditions:
    issues.append(f"Unexpected condition labels in eye metrics: {unexpected_conditions}")
else:
    print("Condition labels OK:", sorted(df_eye["Condition"].dropna().unique()))


# ── 5. MISSING VALUES ────────────────────────────────────────
print("\n=== MISSING VALUES ===")
q_nulls   = df_q.isnull().sum()
eye_nulls = df_eye.isnull().sum()

q_cols_with_nulls   = q_nulls[q_nulls > 0]
eye_cols_with_nulls = eye_nulls[eye_nulls > 0]

if not q_cols_with_nulls.empty:
    issues.append(f"Questionnaire NaNs detected:\n{q_cols_with_nulls}")
    print("Questionnaire NaNs:\n", q_cols_with_nulls)
else:
    print("Questionnaire: no missing values")

if not eye_cols_with_nulls.empty:
    issues.append(f"Eye metrics NaNs detected:\n{eye_cols_with_nulls}")
    print("Eye metrics NaNs:\n", eye_cols_with_nulls)
else:
    print("Eye metrics: no missing values")


# ── 6. DTYPE SANITY ─────────────────────────────────────────
print("\n=== DTYPE CHECKS ===")
# Participant number should be int, age float, scores numeric
EXPECTED_NUMERIC_Q = ["Participant number", "Demographic age"]
for col in EXPECTED_NUMERIC_Q:
    if col in df_q.columns and not pd.api.types.is_numeric_dtype(df_q[col]):
        issues.append(f"Questionnaire column '{col}' expected numeric, got {df_q[col].dtype}")

EXPECTED_NUMERIC_EYE = ["Saccade count", "Fixation count", "Blink count",
                         "Reading time", "Comprehension% (questions correct)"]
for col in EXPECTED_NUMERIC_EYE:
    if col in df_eye.columns and not pd.api.types.is_numeric_dtype(df_eye[col]):
        issues.append(f"Eye metrics column '{col}' expected numeric, got {df_eye[col].dtype}")

print("Dtypes OK for checked columns" if not any("expected numeric" in i for i in issues) else "Dtype issues found — see summary")


# ── 7. VALUE RANGE SANITY ────────────────────────────────────
print("\n=== VALUE RANGES ===")
# VAS scores (questionnaire text questionnaire fields) should be 0–100
vas_cols = [c for c in df_q.columns if "Text questionnaire" in c]
for col in vas_cols:
    if df_q[col].between(0, 100).all() == False:
        issues.append(f"VAS column '{col}' has values outside [0, 100]")

# Comprehension should be 0–100%
if not df_eye["Comprehension% (questions correct)"].dropna().between(0, 100).all():
    issues.append("Comprehension% has values outside [0, 100]")

# Reading time and saccade counts should be positive
for col in ["Reading time", "Saccade count", "Fixation count"]:
    if (df_eye[col] < 0).any():
        issues.append(f"Eye metrics '{col}' has negative values")

print("Value ranges checked")


# ── 8. SUMMARY ───────────────────────────────────────────────
print("\n" + "="*50)
if issues:
    print(f"⚠  {len(issues)} ISSUE(S) FOUND:")
    for i, msg in enumerate(issues, 1):
        print(f"  [{i}] {msg}")
else:
    print("✓  All checks passed — data looks consistent")
print("="*50)

=== SHAPE ===
Questionnaire : 0 rows × 2 cols
Eye metrics   : 75 rows × 19 cols

=== PARTICIPANT ID ALIGNMENT ===
Shared participants : 0
Only in questionnaire : none
Only in eye metrics   : ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'P010', 'P011', 'P012', 'P013', 'P014']

=== CONDITION COMPLETENESS ===
Condition counts per participant:
Condition       eyetracked  full  none
Participant ID                        
P001                    12    12    12
P002                     1     1     1
P003                     1     1     1
P004                     1     1     1
P005                     1     1     1
P006                     1     1     1
P007                     1     1     1
P008                     1     1     1
P009                     1     1     1
P010                     1     1     1
P011                     1     1     1
P012                     1     1     1
P013                     1     1     1
P014                     1     1     1

=== C

***Reading section***
* Reading time RM-anova
* Reading comprehension accuracy RM-anova with Greenhouse-Geisser if Mauchly denies sphericity
* Saccades count RM-anova
* Saccades duration RM-anova
* Fixations count RM-anova
* Fixations duration RM-anova
* Blinks count RM-anova

*Reading time RM-anova*

In [131]:
df_rt = df_eye[["Participant ID", "Condition", "Reading time"]].dropna()
CONDITIONS = sorted(df_rt["Condition"].unique())

result = af.run_rm_anova_or_friedman(
    df            = df_rt,
    subject_col   = "Participant ID",
    condition_col = "Condition",
    value_col     = "Reading time",
    alpha         = 0.05
)

ac = result["assumptions_checked"]
sph = ac.get("sphericity_details", {})

# ── [ANALYSIS] block ─────────────────────────────────────────
print(f"[ANALYSIS] Reading Time by Condition")
print(f"[ANALYSIS] Subjects: {result['n_subjects_final']} used, "
      f"{result['n_subjects_initial'] - result['n_subjects_final']} dropped "
      f"(listwise deletion of incomplete cases)")
print(f"[ANALYSIS] Conditions: {CONDITIONS}")
if result["dropped_subjects"]:
    print(f"[ANALYSIS] Dropped subjects: {result['dropped_subjects']}")

# ── [ASSUMPTION] block ───────────────────────────────────────
print()
print(f"[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes = [n for n in ac["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes:
    for n in normality_notes:
        print(f"[ASSUMPTION]   {n}")
else:
    print(f"[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print(f"[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph.get('sphericity_assumed', 'unknown')}")

if ac["normality_violated"] or not ac["sphericity_verified"]:
    print(f"[ASSUMPTION] One or more assumptions violated — falling back to Friedman test")
else:
    print(f"[ASSUMPTION] All assumptions met — proceeding with RM ANOVA")

# ── [RESULT] block ───────────────────────────────────────────
print()
print(f"[RESULT] Test used: {result['test']}")

if result["test"] == "RM ANOVA":
    df_b, df_e = result["df"]
    print(f"[RESULT] F({df_b}, {df_e}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")
else:
    print(f"[RESULT] chi2({result['df']}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")

if result["significant"]:
    print(f"[RESULT] Significant effect of condition on reading time (p < 0.05)")
else:
    print(f"[RESULT] No significant effect of condition on reading time (p >= 0.05)")

# ── [RESULT] post-hoc ────────────────────────────────────────
if result["post_hoc"] is not None:
    ph = result["post_hoc"]
    label_map = {
        "g1 vs g2": f"{CONDITIONS[0]} vs {CONDITIONS[1]}",
        "g1 vs g3": f"{CONDITIONS[0]} vs {CONDITIONS[2]}",
        "g2 vs g3": f"{CONDITIONS[1]} vs {CONDITIONS[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph['method']}, Bonferroni alpha = {ph['bonferroni_alpha']:.4f}")
    for pair, vals in ph["comparisons"].items():
        label    = label_map.get(pair, pair)
        stat_key = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig      = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print(f"[RESULT] No post-hoc tests run (main test not significant)")

[ANALYSIS] Reading Time by Condition
[ANALYSIS] Subjects: 14 used, 0 dropped (listwise deletion of incomplete cases)
[ANALYSIS] Conditions: ['eyetracked', 'full', 'none']

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.935792, p = 0.671547
[ASSUMPTION]   Sphericity assumed: True
[ASSUMPTION] All assumptions met — proceeding with RM ANOVA

[RESULT] Test used: RM ANOVA
[RESULT] F(2, 26) = 36.2286, p = 0.0000
[RESULT] Significant effect of condition on reading time (p < 0.05)

[RESULT] Post-hoc: Paired t-test with Bonferroni correction, Bonferroni alpha = 0.0167
[RESULT]   eyetracked vs full: t = 6.8094, p = 0.0000 (significant)
[RESULT]   eyetracked vs none: t = 7.0914, p = 0.0000 (significant)
[RESULT]   full vs none: t = 0.1217, p = 0.9050 (not significant)


*Reading comprehension question results RM anova*

In [132]:
# Reading Comprehension Accuracy RM-ANOVA / Friedman fallback
# Accuracy is 0, 33.3, 66.6, or 100.0 (% of 3 MCQs correct per condition)

df_comp = df_eye[["Participant ID", "Condition", "Comprehension% (questions correct)"]].copy()
df_comp["Comprehension% (questions correct)"] = pd.to_numeric(
    df_comp["Comprehension% (questions correct)"], errors="coerce"
)
df_comp = df_comp.dropna(subset=["Participant ID", "Condition", "Comprehension% (questions correct)"])
CONDITIONS = sorted(df_comp["Condition"].unique())

# Descriptive statistics per condition
print("[ANALYSIS] Reading Comprehension Accuracy by Condition")
print("[ANALYSIS] Scale: 0%, 33.3%, 66.6%, 100.0% (4 MCQs per paragraph)")
print()
print("[ANALYSIS] Descriptive statistics:")
for cond in CONDITIONS:
    vals = df_comp[df_comp["Condition"] == cond]["Comprehension% (questions correct)"]
    print(f"[ANALYSIS]   {cond.upper()}: N={len(vals)}, "
          f"M={vals.mean():.1f}%, SD={vals.std():.1f}%, "
          f"Mdn={vals.median():.1f}%, "
          f"range=[{vals.min():.1f}%, {vals.max():.1f}%]")
print()

result = af.run_rm_anova_or_friedman(
    df            = df_comp,
    subject_col   = "Participant ID",
    condition_col = "Condition",
    value_col     = "Comprehension% (questions correct)",
    alpha         = 0.05
)

ac  = result["assumptions_checked"]
sph = ac.get("sphericity_details", {})

# -- [ASSUMPTION] block -----------------------------------------------
# Note: with only 4 possible values (0/33.3/66.6/100), the Shapiro-Wilk
# normality test may flag violations even with well-distributed data.
# The Friedman fallback handles this appropriately.
print("[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes = [n for n in ac["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes:
    for n in normality_notes:
        print(f"[ASSUMPTION]   {n}")
else:
    print("[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print("[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph.get('sphericity_assumed', 'unknown')}")

if ac["normality_violated"] or not ac["sphericity_verified"]:
    print("[ASSUMPTION] One or more assumptions violated — falling back to Friedman test")
else:
    print("[ASSUMPTION] All assumptions met — proceeding with RM ANOVA")

# -- [RESULT] block ---------------------------------------------------
print()
print(f"[RESULT] Test used: {result['test']}")

if result["test"] == "RM ANOVA":
    df_b, df_e = result["df"]
    print(f"[RESULT] F({df_b}, {df_e}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")
else:
    print(f"[RESULT] chi2({result['df']}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")

if result["significant"]:
    print("[RESULT] Significant effect of condition on comprehension accuracy (p < 0.05)")
else:
    print("[RESULT] No significant effect of condition on comprehension accuracy (p >= 0.05)")

# -- [RESULT] post-hoc ------------------------------------------------
if result["post_hoc"] is not None:
    ph = result["post_hoc"]
    label_map = {
        "g1 vs g2": f"{CONDITIONS[0]} vs {CONDITIONS[1]}",
        "g1 vs g3": f"{CONDITIONS[0]} vs {CONDITIONS[2]}",
        "g2 vs g3": f"{CONDITIONS[1]} vs {CONDITIONS[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph['method']}, Bonferroni alpha = {ph['bonferroni_alpha']:.4f}")
    for pair, vals in ph["comparisons"].items():
        label     = label_map.get(pair, pair)
        stat_key  = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig       = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print("[RESULT] No post-hoc tests run (main test not significant)")

[ANALYSIS] Reading Comprehension Accuracy by Condition
[ANALYSIS] Scale: 0%, 33.3%, 66.6%, 100.0% (4 MCQs per paragraph)

[ANALYSIS] Descriptive statistics:
[ANALYSIS]   EYETRACKED: N=16, M=66.7%, SD=27.2%, Mdn=66.7%, range=[33.3%, 100.0%]
[ANALYSIS]   FULL: N=18, M=50.0%, SD=28.6%, Mdn=50.0%, range=[0.0%, 100.0%]
[ANALYSIS]   NONE: N=20, M=60.0%, SD=23.2%, Mdn=66.7%, range=[33.3%, 100.0%]

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   Normality violated for g1-g2 (Shapiro-Wilk p=0.0460)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.958512, p = 0.775505
[ASSUMPTION]   Sphericity assumed: True
[ASSUMPTION] One or more assumptions violated — falling back to Friedman test

[RESULT] Test used: Friedman
[RESULT] chi2(2) = 2.7143, p = 0.2574
[RESULT] No significant effect of condition on comprehension accuracy (p >= 0.05)
[RESULT] No post-hoc tests run (main test not significant)


*Saccades count RM ANOVA*

In [133]:
# Saccade count RM ANOVA / Friedman fallback

df_sc = df_eye[["Participant ID", "Condition", "Saccade count"]].copy()
df_sc["Saccade count"] = pd.to_numeric(df_sc["Saccade count"], errors="coerce")
df_sc = df_sc.dropna(subset=["Participant ID", "Condition", "Saccade count"])
CONDITIONS = sorted(df_sc["Condition"].unique())

result = af.run_rm_anova_or_friedman(
    df            = df_sc,
    subject_col   = "Participant ID",
    condition_col = "Condition",
    value_col     = "Saccade count",
    alpha         = 0.05
)

ac = result["assumptions_checked"]
sph = ac.get("sphericity_details", {})

# -- [ANALYSIS] block -------------------------------------------------
print(f"[ANALYSIS] Saccade Count by Condition")
print(f"[ANALYSIS] Subjects: {result['n_subjects_final']} used, "
      f"{result['n_subjects_initial'] - result['n_subjects_final']} dropped "
      f"(listwise deletion of incomplete cases)")
print(f"[ANALYSIS] Conditions: {CONDITIONS}")
if result["dropped_subjects"]:
    print(f"[ANALYSIS] Dropped subjects: {result['dropped_subjects']}")

# -- [ASSUMPTION] block -----------------------------------------------
print()
print(f"[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes = [n for n in ac["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes:
    for n in normality_notes:
        print(f"[ASSUMPTION]   {n}")
else:
    print(f"[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print(f"[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph.get('sphericity_assumed', 'unknown')}")

if ac["normality_violated"] or not ac["sphericity_verified"]:
    print(f"[ASSUMPTION] One or more assumptions violated - falling back to Friedman test")
else:
    print(f"[ASSUMPTION] All assumptions met - proceeding with RM ANOVA")

# -- [RESULT] block ---------------------------------------------------
print()
print(f"[RESULT] Test used: {result['test']}")

if result["test"] == "RM ANOVA":
    df_b, df_e = result["df"]
    print(f"[RESULT] F({df_b}, {df_e}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")
else:
    print(f"[RESULT] chi2({result['df']}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")

if result["significant"]:
    print(f"[RESULT] Significant effect of condition on saccade count (p < 0.05)")
else:
    print(f"[RESULT] No significant effect of condition on saccade count (p >= 0.05)")

# -- [RESULT] post-hoc ------------------------------------------------
if result["post_hoc"] is not None:
    ph = result["post_hoc"]
    label_map = {
        "g1 vs g2": f"{CONDITIONS[0]} vs {CONDITIONS[1]}",
        "g1 vs g3": f"{CONDITIONS[0]} vs {CONDITIONS[2]}",
        "g2 vs g3": f"{CONDITIONS[1]} vs {CONDITIONS[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph['method']}, Bonferroni alpha = {ph['bonferroni_alpha']:.4f}")
    for pair, vals in ph["comparisons"].items():
        label = label_map.get(pair, pair)
        stat_key = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print(f"[RESULT] No post-hoc tests run (main test not significant)")

[ANALYSIS] Saccade Count by Condition
[ANALYSIS] Subjects: 14 used, 0 dropped (listwise deletion of incomplete cases)
[ANALYSIS] Conditions: ['eyetracked', 'full', 'none']

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   Normality violated for g1-g3 (Shapiro-Wilk p=0.0231)
[ASSUMPTION]   Normality violated for g2-g3 (Shapiro-Wilk p=0.0036)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.885719, p = 0.482810
[ASSUMPTION]   Sphericity assumed: True
[ASSUMPTION] One or more assumptions violated - falling back to Friedman test

[RESULT] Test used: Friedman
[RESULT] chi2(2) = 18.4286, p = 0.0001
[RESULT] Significant effect of condition on saccade count (p < 0.05)

[RESULT] Post-hoc: Wilcoxon signed-rank test with Bonferroni correction, Bonferroni alpha = 0.0167
[RESULT]   eyetracked vs full: W = 7.0000, p = 0.0023 (significant)
[RESULT]   eyetracked vs none: W = 0.0000, p = 0.0001 (significant)
[RESULT]   full vs none: W = 36.0000, p = 0.3258 (no

*Saccades duration RM ANOVA*

In [134]:
df_sd = df_eye[["Participant ID", "Condition", "Saccade avg duration"]].copy()
df_sd["Saccade avg duration"] = pd.to_numeric(df_sd["Saccade avg duration"], errors="coerce")
df_sd = df_sd.dropna(subset=["Participant ID", "Condition", "Saccade avg duration"])
CONDITIONS = sorted(df_sd["Condition"].unique())

result = af.run_rm_anova_or_friedman(
    df            = df_sd,
    subject_col   = "Participant ID",
    condition_col = "Condition",
    value_col     = "Saccade avg duration",
    alpha         = 0.05
)

ac = result["assumptions_checked"]
sph = ac.get("sphericity_details", {})

# -- [ANALYSIS] block -------------------------------------------------
print(f"[ANALYSIS] Saccade Average Duration by Condition")
print(f"[ANALYSIS] Subjects: {result['n_subjects_final']} used, "
      f"{result['n_subjects_initial'] - result['n_subjects_final']} dropped "
      f"(listwise deletion of incomplete cases)")
print(f"[ANALYSIS] Conditions: {CONDITIONS}")
if result["dropped_subjects"]:
    print(f"[ANALYSIS] Dropped subjects: {result['dropped_subjects']}")

# -- [ASSUMPTION] block -----------------------------------------------
print()
print(f"[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes = [n for n in ac["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes:
    for n in normality_notes:
        print(f"[ASSUMPTION]   {n}")
else:
    print(f"[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print(f"[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph.get('sphericity_assumed', 'unknown')}")

if ac["normality_violated"] or not ac["sphericity_verified"]:
    print(f"[ASSUMPTION] One or more assumptions violated - falling back to Friedman test")
else:
    print(f"[ASSUMPTION] All assumptions met - proceeding with RM ANOVA")

# -- [RESULT] block ---------------------------------------------------
print()
print(f"[RESULT] Test used: {result['test']}")

if result["test"] == "RM ANOVA":
    df_b, df_e = result["df"]
    print(f"[RESULT] F({df_b}, {df_e}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")
else:
    print(f"[RESULT] chi2({result['df']}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")

if result["significant"]:
    print(f"[RESULT] Significant effect of condition on saccade avg duration (p < 0.05)")
else:
    print(f"[RESULT] No significant effect of condition on saccade avg duration (p >= 0.05)")

# -- [RESULT] post-hoc ------------------------------------------------
if result["post_hoc"] is not None:
    ph = result["post_hoc"]
    label_map = {
        "g1 vs g2": f"{CONDITIONS[0]} vs {CONDITIONS[1]}",
        "g1 vs g3": f"{CONDITIONS[0]} vs {CONDITIONS[2]}",
        "g2 vs g3": f"{CONDITIONS[1]} vs {CONDITIONS[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph['method']}, Bonferroni alpha = {ph['bonferroni_alpha']:.4f}")
    for pair, vals in ph["comparisons"].items():
        label = label_map.get(pair, pair)
        stat_key = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print(f"[RESULT] No post-hoc tests run (main test not significant)")

[ANALYSIS] Saccade Average Duration by Condition
[ANALYSIS] Subjects: 14 used, 0 dropped (listwise deletion of incomplete cases)
[ANALYSIS] Conditions: ['eyetracked', 'full', 'none']

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.806036, p = 0.274237
[ASSUMPTION]   Sphericity assumed: True
[ASSUMPTION] All assumptions met - proceeding with RM ANOVA

[RESULT] Test used: RM ANOVA
[RESULT] F(2, 26) = 0.2299, p = 0.7962
[RESULT] No significant effect of condition on saccade avg duration (p >= 0.05)
[RESULT] No post-hoc tests run (main test not significant)


*Fixation count RM ANOVA*

In [135]:
# Fixation count RM ANOVA / Friedman fallback

df_fc = df_eye[["Participant ID", "Condition", "Fixation count"]].copy()
df_fc["Fixation count"] = pd.to_numeric(df_fc["Fixation count"], errors="coerce")
df_fc = df_fc.dropna(subset=["Participant ID", "Condition", "Fixation count"])
CONDITIONS = sorted(df_fc["Condition"].unique())

result = af.run_rm_anova_or_friedman(
    df            = df_fc,
    subject_col   = "Participant ID",
    condition_col = "Condition",
    value_col     = "Fixation count",
    alpha         = 0.05
)

ac = result["assumptions_checked"]
sph = ac.get("sphericity_details", {})

# -- [ANALYSIS] block -------------------------------------------------
print(f"[ANALYSIS] Fixation Count by Condition")
print(f"[ANALYSIS] Subjects: {result['n_subjects_final']} used, "
      f"{result['n_subjects_initial'] - result['n_subjects_final']} dropped "
      f"(listwise deletion of incomplete cases)")
print(f"[ANALYSIS] Conditions: {CONDITIONS}")
if result["dropped_subjects"]:
    print(f"[ANALYSIS] Dropped subjects: {result['dropped_subjects']}")

# -- [ASSUMPTION] block -----------------------------------------------
print()
print(f"[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes = [n for n in ac["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes:
    for n in normality_notes:
        print(f"[ASSUMPTION]   {n}")
else:
    print(f"[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print(f"[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph.get('sphericity_assumed', 'unknown')}")

if ac["normality_violated"] or not ac["sphericity_verified"]:
    print(f"[ASSUMPTION] One or more assumptions violated - falling back to Friedman test")
else:
    print(f"[ASSUMPTION] All assumptions met - proceeding with RM ANOVA")

# -- [RESULT] block ---------------------------------------------------
print()
print(f"[RESULT] Test used: {result['test']}")

if result["test"] == "RM ANOVA":
    df_b, df_e = result["df"]
    print(f"[RESULT] F({df_b}, {df_e}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")
else:
    print(f"[RESULT] chi2({result['df']}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")

if result["significant"]:
    print(f"[RESULT] Significant effect of condition on fixation count (p < 0.05)")
else:
    print(f"[RESULT] No significant effect of condition on fixation count (p >= 0.05)")

# -- [RESULT] post-hoc ------------------------------------------------
if result["post_hoc"] is not None:
    ph = result["post_hoc"]
    label_map = {
        "g1 vs g2": f"{CONDITIONS[0]} vs {CONDITIONS[1]}",
        "g1 vs g3": f"{CONDITIONS[0]} vs {CONDITIONS[2]}",
        "g2 vs g3": f"{CONDITIONS[1]} vs {CONDITIONS[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph['method']}, Bonferroni alpha = {ph['bonferroni_alpha']:.4f}")
    for pair, vals in ph["comparisons"].items():
        label = label_map.get(pair, pair)
        stat_key = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print(f"[RESULT] No post-hoc tests run (main test not significant)")

[ANALYSIS] Fixation Count by Condition
[ANALYSIS] Subjects: 14 used, 0 dropped (listwise deletion of incomplete cases)
[ANALYSIS] Conditions: ['eyetracked', 'full', 'none']

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.603937, p = 0.048523
[ASSUMPTION]   Sphericity assumed: False
[ASSUMPTION] One or more assumptions violated - falling back to Friedman test

[RESULT] Test used: Friedman
[RESULT] chi2(2) = 4.6182, p = 0.0994
[RESULT] No significant effect of condition on fixation count (p >= 0.05)
[RESULT] No post-hoc tests run (main test not significant)


*Fixation duration RM ANOVA*

In [136]:
# Fixation duration RM ANOVA / Friedman fallback

df_fd = df_eye[["Participant ID", "Condition", "Fixation avg duration"]].copy()
df_fd["Fixation avg duration"] = pd.to_numeric(df_fd["Fixation avg duration"], errors="coerce")
df_fd = df_fd.dropna(subset=["Participant ID", "Condition", "Fixation avg duration"])
CONDITIONS = sorted(df_fd["Condition"].unique())

result = af.run_rm_anova_or_friedman(
    df            = df_fd,
    subject_col   = "Participant ID",
    condition_col = "Condition",
    value_col     = "Fixation avg duration",
    alpha         = 0.05
)

ac = result["assumptions_checked"]
sph = ac.get("sphericity_details", {})

# -- [ANALYSIS] block -------------------------------------------------
print(f"[ANALYSIS] Fixation Average Duration by Condition")
print(f"[ANALYSIS] Subjects: {result['n_subjects_final']} used, "
      f"{result['n_subjects_initial'] - result['n_subjects_final']} dropped "
      f"(listwise deletion of incomplete cases)")
print(f"[ANALYSIS] Conditions: {CONDITIONS}")
if result["dropped_subjects"]:
    print(f"[ANALYSIS] Dropped subjects: {result['dropped_subjects']}")

# -- [ASSUMPTION] block -----------------------------------------------
print()
print(f"[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes = [n for n in ac["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes:
    for n in normality_notes:
        print(f"[ASSUMPTION]   {n}")
else:
    print(f"[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print(f"[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph.get('sphericity_assumed', 'unknown')}")

if ac["normality_violated"] or not ac["sphericity_verified"]:
    print(f"[ASSUMPTION] One or more assumptions violated - falling back to Friedman test")
else:
    print(f"[ASSUMPTION] All assumptions met - proceeding with RM ANOVA")

# -- [RESULT] block ---------------------------------------------------
print()
print(f"[RESULT] Test used: {result['test']}")

if result["test"] == "RM ANOVA":
    df_b, df_e = result["df"]
    print(f"[RESULT] F({df_b}, {df_e}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")
else:
    print(f"[RESULT] chi2({result['df']}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")

if result["significant"]:
    print(f"[RESULT] Significant effect of condition on fixation avg duration (p < 0.05)")
else:
    print(f"[RESULT] No significant effect of condition on fixation avg duration (p >= 0.05)")

# -- [RESULT] post-hoc ------------------------------------------------
if result["post_hoc"] is not None:
    ph = result["post_hoc"]
    label_map = {
        "g1 vs g2": f"{CONDITIONS[0]} vs {CONDITIONS[1]}",
        "g1 vs g3": f"{CONDITIONS[0]} vs {CONDITIONS[2]}",
        "g2 vs g3": f"{CONDITIONS[1]} vs {CONDITIONS[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph['method']}, Bonferroni alpha = {ph['bonferroni_alpha']:.4f}")
    for pair, vals in ph["comparisons"].items():
        label = label_map.get(pair, pair)
        stat_key = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print(f"[RESULT] No post-hoc tests run (main test not significant)")

[ANALYSIS] Fixation Average Duration by Condition
[ANALYSIS] Subjects: 14 used, 0 dropped (listwise deletion of incomplete cases)
[ANALYSIS] Conditions: ['eyetracked', 'full', 'none']

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   Normality violated for g1-g2 (Shapiro-Wilk p=0.0383)
[ASSUMPTION]   Normality violated for g1-g3 (Shapiro-Wilk p=0.0001)
[ASSUMPTION]   Normality violated for g2-g3 (Shapiro-Wilk p=0.0203)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.629695, p = 0.062342
[ASSUMPTION]   Sphericity assumed: True
[ASSUMPTION] One or more assumptions violated - falling back to Friedman test

[RESULT] Test used: Friedman
[RESULT] chi2(2) = 1.0000, p = 0.6065
[RESULT] No significant effect of condition on fixation avg duration (p >= 0.05)
[RESULT] No post-hoc tests run (main test not significant)


*Blinks count RM anova*

In [137]:
# Blink count RM ANOVA / Friedman fallback

df_bc = df_eye[["Participant ID", "Condition", "Blink count"]].copy()
df_bc["Blink count"] = pd.to_numeric(df_bc["Blink count"], errors="coerce")
df_bc = df_bc.dropna(subset=["Participant ID", "Condition", "Blink count"])
CONDITIONS = sorted(df_bc["Condition"].unique())

result = af.run_rm_anova_or_friedman(
    df            = df_bc,
    subject_col   = "Participant ID",
    condition_col = "Condition",
    value_col     = "Blink count",
    alpha         = 0.05
)

ac = result["assumptions_checked"]
sph = ac.get("sphericity_details", {})

# -- [ANALYSIS] block -------------------------------------------------
print(f"[ANALYSIS] Blink Count by Condition")
print(f"[ANALYSIS] Subjects: {result['n_subjects_final']} used, "
      f"{result['n_subjects_initial'] - result['n_subjects_final']} dropped "
      f"(listwise deletion of incomplete cases)")
print(f"[ANALYSIS] Conditions: {CONDITIONS}")
if result["dropped_subjects"]:
    print(f"[ANALYSIS] Dropped subjects: {result['dropped_subjects']}")

# -- [ASSUMPTION] block -----------------------------------------------
print()
print(f"[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes = [n for n in ac["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes:
    for n in normality_notes:
        print(f"[ASSUMPTION]   {n}")
else:
    print(f"[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print(f"[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph.get('sphericity_assumed', 'unknown')}")

if ac["normality_violated"] or not ac["sphericity_verified"]:
    print(f"[ASSUMPTION] One or more assumptions violated - falling back to Friedman test")
else:
    print(f"[ASSUMPTION] All assumptions met - proceeding with RM ANOVA")

# -- [RESULT] block ---------------------------------------------------
print()
print(f"[RESULT] Test used: {result['test']}")

if result["test"] == "RM ANOVA":
    df_b, df_e = result["df"]
    print(f"[RESULT] F({df_b}, {df_e}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")
else:
    print(f"[RESULT] chi2({result['df']}) = {result['statistic']:.4f}, p = {result['p_value']:.4f}")

if result["significant"]:
    print(f"[RESULT] Significant effect of condition on blink count (p < 0.05)")
else:
    print(f"[RESULT] No significant effect of condition on blink count (p >= 0.05)")

# -- [RESULT] post-hoc ------------------------------------------------
if result["post_hoc"] is not None:
    ph = result["post_hoc"]
    label_map = {
        "g1 vs g2": f"{CONDITIONS[0]} vs {CONDITIONS[1]}",
        "g1 vs g3": f"{CONDITIONS[0]} vs {CONDITIONS[2]}",
        "g2 vs g3": f"{CONDITIONS[1]} vs {CONDITIONS[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph['method']}, Bonferroni alpha = {ph['bonferroni_alpha']:.4f}")
    for pair, vals in ph["comparisons"].items():
        label = label_map.get(pair, pair)
        stat_key = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print(f"[RESULT] No post-hoc tests run (main test not significant)")

[ANALYSIS] Blink Count by Condition
[ANALYSIS] Subjects: 14 used, 0 dropped (listwise deletion of incomplete cases)
[ANALYSIS] Conditions: ['eyetracked', 'full', 'none']

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   Normality violated for g2-g3 (Shapiro-Wilk p=0.0099)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.997423, p = 0.984638
[ASSUMPTION]   Sphericity assumed: True
[ASSUMPTION] One or more assumptions violated - falling back to Friedman test

[RESULT] Test used: Friedman
[RESULT] chi2(2) = 2.7037, p = 0.2588
[RESULT] No significant effect of condition on blink count (p >= 0.05)
[RESULT] No post-hoc tests run (main test not significant)


**Detectability section**
* Identification accuracy Chi-square 3x3 confusion matrix
* Identification response time (ms) RM-anova

In [138]:
from scipy.stats import chi2_contingency

logs_dir = Path("../logs")
detectability_files = sorted(logs_dir.glob("*/*/detectability_trials.csv"))

print(f"Found {len(detectability_files)} detectability_trials.csv files")

# Load and concatenate all files
dfs_detectability = []
for fpath in detectability_files:
    df = pd.read_csv(fpath)
    dfs_detectability.append(df)

df_detect = pd.concat(dfs_detectability, ignore_index=True)
print(f"Loaded {len(df_detect)} total trials")

# Filter main block trials only (is_main_block == True)
df_detect_main = df_detect[df_detect["is_main_block"] == True].copy()
print(f"Main block trials: {len(df_detect_main)}")

# Get unique conditions (should be none, full, eyetracked)
conditions_list = sorted(df_detect_main["condition"].unique())
print(f"Conditions: {conditions_list}")


# Build 3x3 confusion matrix (actual condition vs selected condition)
# Rows = actual condition, Columns = selected condition
confusion_matrix = pd.crosstab(
    df_detect_main["condition"],
    df_detect_main["selected_condition"],
    margins=False
)

# Ensure all conditions are present (handle missing categories)
for cond in conditions_list:
    if cond not in confusion_matrix.index:
        confusion_matrix.loc[cond] = 0
    if cond not in confusion_matrix.columns:
        confusion_matrix[cond] = 0

confusion_matrix = confusion_matrix.reindex(index=conditions_list, columns=conditions_list, fill_value=0)
print("\n[ANALYSIS] Detectability - Identification Accuracy (3x3 Confusion Matrix)")
print("[ANALYSIS] Rows = Actual Condition | Columns = Selected Condition")
print()
print(confusion_matrix)


# Overall chi-square test on the full 3x3 confusion matrix
# H0: Response distribution is independent of actual condition (no classification ability)
# H1: Response distribution depends on actual condition (participants can discriminate)
print()
print("[ANALYSIS] Chi-Square Test of Independence — Full 3x3 Confusion Matrix")
print("[ANALYSIS] H0: Responses are independent of condition (no discrimination ability)")
print("[ANALYSIS] H1: Responses depend on condition (participants can discriminate)")
print()

chi2, chi2_p, dof, expected = chi2_contingency(confusion_matrix)
chi2_sig = "Reject H0" if chi2_p < alpha else "Fail to reject H0"
print(f"[RESULT] χ²({dof}) = {chi2:.4f}, p = {chi2_p:.6f} ({chi2_sig})")
print()
print("[ANALYSIS] Expected cell frequencies under H0 (chance responding):")
expected_df = pd.DataFrame(expected, index=conditions_list, columns=conditions_list)
print(expected_df.round(2))


# Per-condition exact binomial test
# H0: Identification accuracy = 33.3% (chance for 3-category task)
# H1: Identification accuracy > 33.3% (participants perform above chance)
print()
print("[ANALYSIS] Exact Binomial Test — Per Condition")
print("[ANALYSIS] H0: Accuracy = 33.3% (chance) | H1: Accuracy > 33.3%")
print()

alpha = 0.05
all_results = []

for condition in conditions_list:
    row = confusion_matrix.loc[condition]
    total_trials = int(row.sum())
    correct_count = int(row.get(condition, 0))
    incorrect_count = total_trials - correct_count
    accuracy_pct = (correct_count / total_trials * 100) if total_trials else float("nan")

    # p=1/3 reflects chance for a 3-alternative forced-choice task
    # alternative="greater" tests whether accuracy exceeds chance
    binom_result = stats.binomtest(correct_count, n=total_trials, p=1/3, alternative="greater")

    p_value = binom_result.pvalue
    statistic = binom_result.statistic

    all_results.append({
        "Condition": condition,
        "Test": "Exact binomial test",
        "Correct_count": correct_count,
        "Incorrect_count": incorrect_count,
        "p_value": p_value,
        "Significant": p_value < alpha,
        "accuracy": accuracy_pct
    })

    sig = "Reject H0" if p_value < alpha else "Fail to reject H0"
    print(f"[RESULT] {condition.upper()} condition:")
    print(f"[RESULT]   Observed responses: [{correct_count} correct, {incorrect_count} incorrect]")
    print(f"[RESULT]   Binomial test: statistic = {statistic:.4f}, p = {p_value:.6f} ({sig})")
    print(f"[RESULT]   Identification accuracy: {accuracy_pct:.1f}% (chance = 33.3%)")
    print()

# Summary
print("[RESULT] SUMMARY:")
above_chance = [r for r in all_results if r["Significant"]]
if above_chance:
    cond_names = [r["Condition"] for r in above_chance]
    print(f"[RESULT]   Conditions identified above chance: {', '.join(cond_names)}")
    print(f"[RESULT]   (reject H0 for these conditions, p < {alpha})")
else:
    print(f"[RESULT]   No condition was identified above chance level")
    print(f"[RESULT]   (all p-values >= {alpha})")


# Accuracy per participant per condition
participant_accuracy = (
    df_detect_main
    .assign(correct=lambda d: d["condition"] == d["selected_condition"])
    .groupby(["participant_id", "condition"])["correct"]
    .mean()
    .reset_index()
)

# Per-condition: one-sample t-test against chance (0.333)
# Now N = number of participants, not number of trials
from scipy.stats import ttest_1samp

for condition in conditions_list:
    acc = participant_accuracy[participant_accuracy["condition"] == condition]["correct"]
    t, p = ttest_1samp(acc, popmean=1/3)
    print(f"{condition}: M = {acc.mean():.3f}, t({len(acc)-1}) = {t:.3f}, p = {p:.4f}")

Found 14 detectability_trials.csv files
Loaded 588 total trials
Main block trials: 504
Conditions: ['eyetracked', 'full', 'none']

[ANALYSIS] Detectability - Identification Accuracy (3x3 Confusion Matrix)
[ANALYSIS] Rows = Actual Condition | Columns = Selected Condition

selected_condition  eyetracked  full  none
condition                                 
eyetracked                 159     0     9
full                         1   166     1
none                         0     0   168

[ANALYSIS] Chi-Square Test of Independence — Full 3x3 Confusion Matrix
[ANALYSIS] H0: Responses are independent of condition (no discrimination ability)
[ANALYSIS] H1: Responses depend on condition (participants can discriminate)

[RESULT] χ²(4) = 945.1049, p = 0.000000 (Reject H0)

[ANALYSIS] Expected cell frequencies under H0 (chance responding):


            eyetracked   full   none
eyetracked       53.33  55.33  59.33
full             53.33  55.33  59.33
none             53.33  55.33  59.33

[ANALYSIS] Exact Binomial Test — Per Condition
[ANALYSIS] H0: Accuracy = 33.3% (chance) | H1: Accuracy > 33.3%

[RESULT] EYETRACKED condition:
[RESULT]   Observed responses: [159 correct, 9 incorrect]
[RESULT]   Binomial test: statistic = 0.9464, p = 0.000000 (Reject H0)
[RESULT]   Identification accuracy: 94.6% (chance = 33.3%)

[RESULT] FULL condition:
[RESULT]   Observed responses: [166 correct, 2 incorrect]
[RESULT]   Binomial test: statistic = 0.9881, p = 0.000000 (Reject H0)
[RESULT]   Identification accuracy: 98.8% (chance = 33.3%)

[RESULT] NONE condition:
[RESULT]   Observed responses: [168 correct, 0 incorrect]
[RESULT]   Binomial test: statistic = 1.0000, p = 0.000000 (Reject H0)
[RESULT]   Identification accuracy: 100.0% (chance = 33.3%)

[RESULT] SUMMARY:
[RESULT]   Conditions identified above chance: eyetracked, full, none
[R

c:\Users\Bruger\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:523: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [139]:
# Identification Response Time RM ANOVA / Friedman fallback

df_rt_detect = df_detect_main[["participant_id", "condition", "rt_ms"]].copy()
df_rt_detect["rt_ms"] = pd.to_numeric(df_rt_detect["rt_ms"], errors="coerce")
df_rt_detect = df_rt_detect.dropna(subset=["participant_id", "condition", "rt_ms"])

# Rename participant_id column to match the function's expected "Participant ID"
df_rt_detect.rename(columns={"participant_id": "Participant ID"}, inplace=True)

CONDITIONS_DETECT = sorted(df_rt_detect["condition"].unique())

result_rt = af.run_rm_anova_or_friedman(
    df            = df_rt_detect,
    subject_col   = "Participant ID",
    condition_col = "condition",
    value_col     = "rt_ms",
    alpha         = 0.05
)

ac_rt = result_rt["assumptions_checked"]
sph_rt = ac_rt.get("sphericity_details", {})

# -- [ANALYSIS] block -------------------------------------------------
print(f"[ANALYSIS] Identification Response Time by Condition")
print(f"[ANALYSIS] Subjects: {result_rt['n_subjects_final']} used, "
      f"{result_rt['n_subjects_initial'] - result_rt['n_subjects_final']} dropped "
      f"(listwise deletion of incomplete cases)")
print(f"[ANALYSIS] Conditions: {CONDITIONS_DETECT}")
if result_rt["dropped_subjects"]:
    print(f"[ANALYSIS] Dropped subjects: {result_rt['dropped_subjects']}")

# -- [ASSUMPTION] block -----------------------------------------------
print()
print(f"[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):")
normality_notes_rt = [n for n in ac_rt["notes"] if "Normality" in n or "normality" in n or "Shapiro" in n]
if normality_notes_rt:
    for n in normality_notes_rt:
        print(f"[ASSUMPTION]   {n}")
else:
    print(f"[ASSUMPTION]   All pairwise differences passed normality (p >= 0.05)")

print(f"[ASSUMPTION] Sphericity (Mauchly's test):")
print(f"[ASSUMPTION]   W = {sph_rt.get('mauchly_stat', float('nan')):.6f}, "
      f"p = {sph_rt.get('mauchly_p', float('nan')):.6f}")
print(f"[ASSUMPTION]   Sphericity assumed: {sph_rt.get('sphericity_assumed', 'unknown')}")

if ac_rt["normality_violated"] or not ac_rt["sphericity_verified"]:
    print(f"[ASSUMPTION] One or more assumptions violated - falling back to Friedman test")
else:
    print(f"[ASSUMPTION] All assumptions met - proceeding with RM ANOVA")

# -- [RESULT] block ---------------------------------------------------
print()
print(f"[RESULT] Main effect - Condition:")
print(f"[RESULT]   Test: {result_rt['test']}")
print(f"[RESULT]   Statistic: {result_rt['statistic']:.4f}")
print(f"[RESULT]   p-value: {result_rt['p_value']:.6f}")

if result_rt['p_value'] < 0.05:
    print(f"[RESULT]   Result: SIGNIFICANT (p < 0.05)")
else:
    print(f"[RESULT]   Result: NOT SIGNIFICANT (p >= 0.05)")

# -- [RESULT] post-hoc ------------------------------------------------
if result_rt["post_hoc"] is not None:
    ph_rt = result_rt["post_hoc"]
    label_map_rt = {
        "g1 vs g2": f"{CONDITIONS_DETECT[0]} vs {CONDITIONS_DETECT[1]}",
        "g1 vs g3": f"{CONDITIONS_DETECT[0]} vs {CONDITIONS_DETECT[2]}",
        "g2 vs g3": f"{CONDITIONS_DETECT[1]} vs {CONDITIONS_DETECT[2]}",
    }
    print()
    print(f"[RESULT] Post-hoc: {ph_rt['method']}, Bonferroni alpha = {ph_rt['bonferroni_alpha']:.4f}")
    for pair, vals in ph_rt["comparisons"].items():
        label = label_map_rt.get(pair, pair)
        stat_key = "t_statistic" if "t_statistic" in vals else "z_statistic"
        stat_name = "t" if "t_statistic" in vals else "W"
        sig = "significant" if vals["significant"] else "not significant"
        print(f"[RESULT]   {label}: {stat_name} = {vals[stat_key]:.4f}, "
              f"p = {vals['p_value']:.4f} ({sig})")
else:
    print(f"[RESULT] No post-hoc tests run (main test not significant)")

[ANALYSIS] Identification Response Time by Condition
[ANALYSIS] Subjects: 14 used, 0 dropped (listwise deletion of incomplete cases)
[ANALYSIS] Conditions: ['eyetracked', 'full', 'none']

[ASSUMPTION] Normality (Shapiro-Wilk on pairwise differences):
[ASSUMPTION]   Normality violated for g1-g2 (Shapiro-Wilk p=0.0475)
[ASSUMPTION] Sphericity (Mauchly's test):
[ASSUMPTION]   W = 0.682915, p = 0.101437
[ASSUMPTION]   Sphericity assumed: True
[ASSUMPTION] One or more assumptions violated - falling back to Friedman test

[RESULT] Main effect - Condition:
[RESULT]   Test: Friedman
[RESULT]   Statistic: 10.7143
[RESULT]   p-value: 0.004714
[RESULT]   Result: SIGNIFICANT (p < 0.05)

[RESULT] Post-hoc: Wilcoxon signed-rank test with Bonferroni correction, Bonferroni alpha = 0.0167
[RESULT]   eyetracked vs full: W = 40.0000, p = 0.4631 (not significant)
[RESULT]   eyetracked vs none: W = 8.0000, p = 0.0031 (significant)
[RESULT]   full vs none: W = 5.0000, p = 0.0012 (significant)


**Questionnaire descriptive and explorative statistics**
Attempting to find patterns across specifics in the questionnaire data, such as trends in eye strain types (whether each condition causes dryness, doubled vision, or similar profiles to each other but at different scales)